In [ ]:
!pip -q install -U groq flask

import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
print("✅ API key stored for this session")

In [ ]:
# ============================================================
#  PROMPT·LAB  —  Electro-Biomorph Prompt Evaluation Studio
#  Colab cell 2 of 2  (no Streamlit: pure Python + HTML/JS)
#  Run cell 1 first (installs Groq + stores your API key).
# ============================================================
import os, re, json, time
from IPython.display import HTML, JSON, display
from google.colab import output
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODELS = {"openai/gpt-oss-20b", "openai/gpt-oss-120b", "llama-3.3-70b-versatile"}
STRICT_SCHEMA_MODELS = {"openai/gpt-oss-20b", "openai/gpt-oss-120b"}

# ---------------- scenarios & criteria (from your notebook) ----------------
SCENARIOS = {
    "Business": {
        "Beginner":     {"title": "Business Email", "description": "Create a professional email explaining a one-week project delay to a client."},
        "Intermediate": {"title": "Meeting Summary", "description": "Convert meeting notes into a structured summary containing decisions, action items, owners and deadlines."},
        "Advanced":     {"title": "Executive Business Report", "description": "Analyze quarterly business performance and create an executive report containing insights, risks, trends and recommendations."},
    },
    "HR": {
        "Beginner":     {"title": "Job Description", "description": "Create a job description for a Python Developer."},
        "Intermediate": {"title": "Interview Questions", "description": "Generate structured interview questions for an AI/ML Engineer."},
        "Advanced":     {"title": "Candidate Evaluation", "description": "Evaluate candidate interview responses against predefined competencies and provide structured feedback."},
    },
    "Marketing": {
        "Beginner":     {"title": "Social Media Post", "description": "Create a LinkedIn post announcing a new technology product."},
        "Intermediate": {"title": "Marketing Campaign", "description": "Design a marketing campaign for a new technology product."},
        "Advanced":     {"title": "Go-To-Market Strategy", "description": "Analyze a target market and create a complete go-to-market strategy."},
    },
    "Data Analytics": {
        "Beginner":     {"title": "Sales Summary", "description": "Summarize monthly sales data."},
        "Intermediate": {"title": "Business Insights", "description": "Analyze sales data and identify trends, patterns and anomalies."},
        "Advanced":     {"title": "Executive Analytics", "description": "Analyze a business dataset and produce actionable executive insights."},
    },
    "Software Development": {
        "Beginner":     {"title": "Python Function", "description": "Generate a Python function to calculate factorial."},
        "Intermediate": {"title": "REST API", "description": "Create a REST API using Python and FastAPI."},
        "Advanced":     {"title": "Code Review", "description": "Perform a detailed code review covering security, performance, maintainability and coding standards."},
    },
}

CRITERIA = {
    "Role": "Does the prompt clearly define who the AI should act as?",
    "Goal": "Does the prompt clearly state what the AI needs to accomplish?",
    "Context": "Does the prompt provide sufficient background information?",
    "Constraints": "Does the prompt define rules, limitations or boundaries?",
    "Format": "Does the prompt specify the expected output structure or format?",
    "Style": "Does the prompt specify the desired tone, style or communication approach?",
}

# ---------------- LLM-as-a-judge ----------------
def _obj(t):
    return {"type": "object",
            "properties": {k: {"type": t} for k in CRITERIA},
            "required": list(CRITERIA), "additionalProperties": False}

EVAL_SCHEMA = {
    "type": "object",
    "properties": {
        "scores": _obj("number"),
        "explanations": _obj("string"),
        "strengths": {"type": "array", "items": {"type": "string"}},
        "weaknesses": {"type": "array", "items": {"type": "string"}},
        "missing_components": {"type": "array", "items": {"type": "string"}},
        "recommendations": {"type": "array", "items": {"type": "string"}},
        "optimized_prompt": {"type": "string"},
    },
    "required": ["scores", "explanations", "strengths", "weaknesses",
                 "missing_components", "recommendations", "optimized_prompt"],
    "additionalProperties": False,
}

def _system_prompt(scenario, level):
    criteria_text = "\n".join(f"{n}: {d}" for n, d in CRITERIA.items())
    return f"""
You are an expert Prompt Engineering evaluator.

Evaluate the user's prompt using these six criteria:

{criteria_text}

Scenario:
{scenario}

Difficulty:
{level}

Scoring scale:
0-2 = Missing | 3-4 = Weak | 5-6 = Basic | 7-8 = Good | 9 = Excellent | 10 = Outstanding

Be objective and constructive.
For every criterion provide a score and an explanation.
Also provide: strengths, weaknesses, missing_components, recommendations, optimized_prompt.
Do NOT calculate the overall score. Python will calculate it.
"""

def _parse_json(text):
    t = (text or "").replace("```json", "").replace("```", "").strip()
    a, b = t.find("{"), t.rfind("}")
    return json.loads(t[a:b + 1])

def _sanitize(r):
    sc = {k: max(0.0, min(10.0, float((r.get("scores") or {}).get(k, 0)))) for k in CRITERIA}
    ex = {k: str((r.get("explanations") or {}).get(k, "")) for k in CRITERIA}
    lst = lambda key: [str(x) for x in (r.get(key) or [])]
    return {
        "scores": sc, "explanations": ex,
        "strengths": lst("strengths"), "weaknesses": lst("weaknesses"),
        "missing_components": lst("missing_components"),
        "recommendations": lst("recommendations"),
        "optimized_prompt": str(r.get("optimized_prompt", "")),
        "overall_score": round(sum(sc.values()) / 60 * 100),   # Python calculates it
    }

def evaluate_prompt(prompt, scenario, level, model):
    messages = [
        {"role": "system", "content": _system_prompt(scenario, level)},
        {"role": "user", "content": f"Evaluate this prompt:\n\n{prompt}"},
    ]
    if model in STRICT_SCHEMA_MODELS:
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=0.2,
                response_format={"type": "json_schema",
                                 "json_schema": {"name": "prompt_evaluation", "strict": True, "schema": EVAL_SCHEMA}})
            return _sanitize(json.loads(resp.choices[0].message.content))
        except Exception:
            pass  # fall through to plain-JSON mode
    messages[0]["content"] += ("\nReturn ONLY valid JSON with keys: scores, explanations (each keyed by "
                               + ", ".join(CRITERIA) + "), strengths, weaknesses, missing_components, "
                               "recommendations, optimized_prompt.")
    resp = client.chat.completions.create(model=model, messages=messages, temperature=0.2)
    return _sanitize(_parse_json(resp.choices[0].message.content))

def run_prompt(prompt, model):
    resp = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        temperature=0.7, max_completion_tokens=1400)
    return resp.choices[0].message.content or "(empty response)"

# ---------------- API functions (used by both the inline UI and the web link) ----------------
def api_evaluate(prompt, domain, level, scenario, model):
    try:
        if model not in MODELS:
            model = "openai/gpt-oss-20b"
        return evaluate_prompt(prompt, scenario or domain, level, model)
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

def api_battle(prompt_a, prompt_b, model):
    try:
        if model not in MODELS:
            model = "openai/gpt-oss-20b"
        return {"a": run_prompt(prompt_a, model), "b": run_prompt(prompt_b, model)}
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}

def api_save(name, content):
    os.makedirs("/content/prompt_studio_exports", exist_ok=True)
    safe = re.sub(r"[^A-Za-z0-9._-]", "_", os.path.basename(str(name))) or "export.txt"
    path = f"/content/prompt_studio_exports/{int(time.time())}_{safe}"
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return {"path": path}

API = {"ps_evaluate": api_evaluate, "ps_battle": api_battle, "ps_save": api_save}
for _name, _fn in API.items():   # inline (notebook output) bridge
    output.register_callback(_name, (lambda f: (lambda *a: JSON(f(*a))))(_fn))

# ---------------- front-end ----------------
UI = r"""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300..700&family=JetBrains+Mono:wght@300;400;600&family=Syne:wght@700;800&display=swap" rel="stylesheet">
<style>
@property --k{syntax:'<number>';inherits:false;initial-value:0}
@property --a{syntax:'<angle>';inherits:false;initial-value:0deg}
#ps{--char:#090b0a;--char-2:#0e1210;--char-3:#151b18;--line:rgba(120,190,150,.15);
 --forest:#14532d;--forest-2:#1f7a45;--moss:#3fcf8e;--crimson:#dc143c;--blood:#8f0f26;--ember:#ff3b5c;
 --text:#e4eee8;--mute:#7e9388;--glow:1;--fs:17px;
 position:relative;width:100%;height:900px;overflow:hidden;border-radius:34px 10px 34px 10px;
 background:radial-gradient(1200px 700px at 15% -10%,#123a25 0%,transparent 60%),radial-gradient(900px 600px at 110% 20%,#3a0a14 0%,transparent 60%),var(--char);
 color:var(--text);font-family:'Space Grotesk',system-ui,sans-serif;box-shadow:0 0 0 1px var(--line),0 30px 80px rgba(0,0,0,.6)}
#ps *{box-sizing:border-box}
#ps button,#ps select,#ps input,#ps textarea{font-family:inherit;color:inherit}
#psBg{position:absolute;inset:0;width:100%;height:100%;z-index:0;pointer-events:none}
#scroller{position:absolute;inset:0;overflow-y:auto;overflow-x:hidden;z-index:1;scroll-behavior:smooth;scrollbar-width:thin;scrollbar-color:var(--crimson) transparent}
#scroller::-webkit-scrollbar{width:6px}#scroller::-webkit-scrollbar-thumb{background:linear-gradient(var(--crimson),var(--forest-2));border-radius:6px}
.top{position:absolute;top:0;left:0;right:0;z-index:20;display:flex;align-items:center;gap:14px;padding:14px 22px;background:linear-gradient(rgba(9,11,10,.92),rgba(9,11,10,0));backdrop-filter:blur(8px);-webkit-mask:linear-gradient(#000 70%,transparent)}
.logo{display:flex;align-items:center;gap:10px;font-family:'Syne',sans-serif;font-weight:800;letter-spacing:.08em;font-size:15px;cursor:default}
.logo svg{width:30px;height:30px;filter:drop-shadow(0 0 calc(8px*var(--glow)) var(--crimson))}
.nav{display:flex;gap:6px;margin-left:auto}
.nav button,.chip,.seg button{background:rgba(20,28,24,.7);border:1px solid var(--line);border-radius:16px 5px 16px 5px;padding:7px 14px;font-size:12.5px;letter-spacing:.06em;text-transform:uppercase;cursor:pointer;transition:.25s}
.nav button:hover,.chip:hover,.seg button:hover{border-color:var(--moss);box-shadow:0 0 calc(14px*var(--glow)) rgba(63,207,142,.35)}
.top select{background:rgba(20,28,24,.8);border:1px solid var(--line);border-radius:12px;padding:7px 10px;font-size:12px;font-family:'JetBrains Mono',monospace}
.top select option{background:#0e1210}
section{position:relative;padding:34px 34px 60px}
#hero{height:820px;display:flex;flex-direction:column;align-items:center;justify-content:center;text-align:center;cursor:none;overflow:hidden}
.blob{position:absolute;filter:blur(46px);mix-blend-mode:screen;animation:morph 16s ease-in-out infinite;opacity:calc(.5*var(--glow));pointer-events:none}
.b1{width:420px;height:420px;left:6%;top:10%;background:radial-gradient(circle at 30% 30%,#2f8f5b,#0f3d24 70%)}
.b2{width:360px;height:360px;right:8%;bottom:6%;background:radial-gradient(circle at 60% 40%,#dc143c,#4a0a16 70%);animation-delay:-6s}
.b3{width:260px;height:260px;left:46%;top:60%;background:radial-gradient(circle,#1f7a45,transparent 70%);animation-delay:-11s}
@keyframes morph{0%,100%{border-radius:42% 58% 63% 37%/41% 44% 56% 59%;transform:translate(0,0) rotate(0)}33%{border-radius:68% 32% 38% 62%/53% 62% 38% 47%;transform:translate(24px,-18px) rotate(40deg)}66%{border-radius:35% 65% 54% 46%/66% 36% 64% 34%;transform:translate(-18px,22px) rotate(-30deg)}}
.eyebrow{font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:.3em;text-transform:uppercase;color:var(--moss);margin-bottom:22px}
.kinetic{font-family:'Syne',sans-serif;font-weight:800;text-transform:uppercase;letter-spacing:-.02em;line-height:.95;perspective:600px;white-space:nowrap}
.hero-l1{font-size:clamp(34px,6.6vw,74px)}.hero-l2{font-size:clamp(44px,9.5vw,112px);margin-top:6px}
.hero-l2 .ch{--hc:var(--moss)}
.ch{display:inline-block;--k:0;--x:0px;--y:0px;--r:0deg;--rx:0deg;--s:1;--o:1;
 transform:translate(var(--x),calc(var(--y) + var(--k)*-10px)) rotateX(var(--rx)) rotate(var(--r)) scale(calc(var(--s) + var(--k)*.28));
 opacity:var(--o);color:var(--hc,var(--text));color:color-mix(in srgb,var(--ember) calc(var(--k)*100%),var(--hc,var(--text)));
 text-shadow:0 0 calc(var(--k)*24px*var(--glow)) rgba(220,20,60,calc(var(--k)*.9));transition:--k .4s cubic-bezier(.2,.8,.2,1);will-change:transform}
.sub{margin-top:26px;max-width:560px;color:var(--mute);font-size:16px;line-height:1.55}
.sub b{color:var(--text);font-weight:600}
.hint{margin-top:34px;font-size:11px;letter-spacing:.2em;text-transform:uppercase;opacity:.5;position:relative;padding-bottom:70px}
.hint::after{content:'';position:absolute;left:50%;top:100%;width:1px;height:64px;margin-top:-64px;background:linear-gradient(transparent,var(--crimson))}
.dot,.orb{position:absolute;left:0;top:0;pointer-events:none;z-index:5}
.dot{width:12px;height:12px;margin:-6px 0 0 -6px;border:1.5px solid var(--moss);border-radius:50%;opacity:0}
.orb{width:46px;height:46px;margin:-23px 0 0 -23px;border-radius:50%;background:radial-gradient(circle at 35% 35%,#ff6b85,var(--crimson) 55%,var(--blood));box-shadow:0 0 calc(30px*var(--glow)) rgba(220,20,60,.7);mix-blend-mode:screen;filter:url(#goo)}
.legend{position:absolute;bottom:18px;left:24px;font-family:'JetBrains Mono',monospace;font-size:10.5px;color:var(--mute);display:flex;gap:16px}
.legend i{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px;vertical-align:-1px}
h2.sec{font-family:'Syne',sans-serif;font-weight:800;font-size:clamp(22px,3.2vw,34px);text-transform:uppercase;letter-spacing:.02em;margin:0 0 6px}
.lede{color:var(--mute);margin:0 0 22px;font-size:14.5px}
.lab{font-family:'JetBrains Mono',monospace;font-size:11px;letter-spacing:.22em;text-transform:uppercase;color:var(--moss);cursor:default;display:inline-block;margin-bottom:10px}
.card{position:relative;background:linear-gradient(160deg,rgba(22,30,26,.88),rgba(11,15,13,.94));border:1px solid var(--line);border-radius:30px 12px 34px 12px;padding:22px;backdrop-filter:blur(6px)}
.edge::before{content:'';position:absolute;inset:-1px;border-radius:inherit;padding:1.5px;background:conic-gradient(from var(--a),transparent 0 55%,var(--crimson) 72%,var(--moss) 88%,transparent);-webkit-mask:linear-gradient(#000 0 0) content-box,linear-gradient(#000 0 0);-webkit-mask-composite:xor;mask-composite:exclude;animation:spin 7s linear infinite;pointer-events:none;opacity:calc(.9*var(--glow))}
@keyframes spin{to{--a:360deg}}
.row{display:flex;flex-wrap:wrap;gap:8px;align-items:center}
.chip.on,.seg button.on{background:linear-gradient(120deg,var(--forest-2),var(--blood));border-color:transparent;box-shadow:0 0 calc(18px*var(--glow)) rgba(220,20,60,.35)}
.seg{display:flex;gap:6px}
.ch-card{margin-top:16px;padding:16px 18px;border-left:3px solid var(--crimson);background:rgba(220,20,60,.06);border-radius:4px 18px 4px 18px}
.ch-card h4{margin:0 0 4px;font-size:17px}.ch-card p{margin:0;color:var(--mute);font-size:14px;line-height:1.5}
.grid2{display:grid;grid-template-columns:1.7fr 1fr;gap:18px}
@media(max-width:820px){.grid2,.grid3{grid-template-columns:1fr!important}}
.caret-wrap{position:relative}
.caret-wrap textarea,.caret-wrap input{width:100%;display:block;background:rgba(5,8,6,.7);border:1px solid var(--line);border-radius:22px 8px 22px 8px;padding:16px 18px;font-size:var(--fs);line-height:1.55;outline:none;resize:vertical;caret-color:transparent;transition:border-color .25s,box-shadow .25s}
.caret-wrap textarea{min-height:230px}
.caret-wrap input{font-size:15px;padding:11px 14px;line-height:1.4}
.caret-wrap textarea:focus,.caret-wrap input:focus{border-color:var(--moss);box-shadow:0 0 calc(24px*var(--glow)) rgba(63,207,142,.22)}
.caret-wrap textarea::placeholder,.caret-wrap input::placeholder{color:rgba(228,238,232,.3)}
.mirror{position:absolute;top:0;left:0;visibility:hidden;overflow:hidden;pointer-events:none;overflow-wrap:break-word}
.scaret{position:absolute;left:0;top:0;width:2.5px;border-radius:3px;pointer-events:none;background:linear-gradient(var(--ember),var(--moss));box-shadow:0 0 calc(12px*var(--glow)) var(--crimson);visibility:hidden}
.scaret.show{visibility:visible}.scaret.idle{animation:blink 1.1s steps(2,start) infinite}
@keyframes blink{50%{opacity:0}}
.stats{display:flex;gap:16px;font-family:'JetBrains Mono',monospace;font-size:11.5px;color:var(--mute);margin:10px 2px 14px}
.stats b{color:var(--text);font-weight:600}
.btn{border:1px solid var(--line);background:rgba(20,28,24,.8);border-radius:18px 6px 18px 6px;padding:11px 18px;font-size:13px;letter-spacing:.07em;text-transform:uppercase;cursor:pointer;transition:.25s;font-weight:600}
.btn:hover{border-color:var(--moss);box-shadow:0 0 calc(16px*var(--glow)) rgba(63,207,142,.3);transform:translateY(-1px)}
.btn.pri{background:linear-gradient(120deg,var(--forest-2),var(--blood) 130%);border-color:transparent}
.btn.pri:hover{box-shadow:0 0 calc(26px*var(--glow)) rgba(220,20,60,.55)}
.btn:disabled{opacity:.6;cursor:wait}
.btn.busy{animation:zap .6s infinite alternate}
@keyframes zap{from{box-shadow:0 0 6px rgba(63,207,142,.2)}to{box-shadow:0 0 30px rgba(220,20,60,.8)}}
.shake{animation:shake .4s}@keyframes shake{25%{transform:translateX(-6px)}75%{transform:translateX(6px)}}
.qs{display:grid;gap:11px}
.qb{display:grid;grid-template-columns:86px 1fr 20px;align-items:center;gap:10px;font-size:12.5px}
.qb .tr{height:7px;border-radius:7px;background:rgba(255,255,255,.07);overflow:hidden}
.qb .fl{height:100%;width:calc(var(--v)*100%);border-radius:7px;background:linear-gradient(90deg,var(--crimson),var(--forest-2));transition:width .5s cubic-bezier(.2,.9,.2,1)}
.qb .ok{opacity:0;color:var(--moss);transition:.3s}.qb.full .ok{opacity:1}
.note{font-size:11.5px;color:var(--mute);margin-top:14px;line-height:1.5}
#results{display:none}#results.on{display:block;animation:rise .7s cubic-bezier(.2,.9,.2,1)}
@keyframes rise{from{opacity:0;transform:translateY(30px)}}
.hdrstats{display:grid;grid-template-columns:220px 1fr;gap:20px;align-items:center}
.gauge-w{position:relative;width:200px;height:200px}
.gauge{width:100%;height:100%}.g-track{fill:none;stroke:rgba(255,255,255,.07);stroke-width:9}
.g-arc{fill:none;stroke-width:9;stroke-linecap:round;transition:stroke-dashoffset 1.2s cubic-bezier(.2,.9,.2,1),stroke .6s;filter:drop-shadow(0 0 calc(6px*var(--glow)) currentColor)}
.g-num{position:absolute;inset:0;display:flex;flex-direction:column;align-items:center;justify-content:center;font-family:'Syne',sans-serif}
.g-num b{font-size:58px;line-height:1}.g-num small{font-size:13px;color:var(--mute);font-family:'JetBrains Mono',monospace}
.rate{font-family:'Syne',sans-serif;font-size:clamp(26px,4vw,40px);font-weight:800;text-transform:uppercase;line-height:1}
.kpis{display:flex;flex-wrap:wrap;gap:10px;margin-top:14px}
.kpi{padding:9px 14px;border:1px solid var(--line);border-radius:14px 4px 14px 4px;font-size:12px;color:var(--mute)}.kpi b{display:block;color:var(--text);font-size:18px}
.up{color:var(--moss)!important}.down{color:var(--ember)!important}
.dchips{display:flex;flex-wrap:wrap;gap:6px;margin-top:12px}.dchips span{font-family:'JetBrains Mono',monospace;font-size:11px;padding:3px 9px;border-radius:10px;background:rgba(255,255,255,.05)}
.grid3{display:grid;grid-template-columns:1fr 1fr;gap:18px;margin-top:18px}
#radar{width:100%;max-width:440px;margin:auto;display:block;overflow:visible}
.rg-ring{fill:none;stroke:rgba(120,190,150,.14)}.rg-ax{stroke:rgba(120,190,150,.2)}.rg-lb{fill:var(--text);font-size:13px;font-family:'Space Grotesk',system-ui,sans-serif;letter-spacing:.05em}
.rg-prev{fill:none;stroke:var(--crimson);stroke-width:1.5;stroke-dasharray:5 4;opacity:.8}
.rg-cur-g{transform-origin:200px 200px;transform:scale(.01);transition:transform 1.1s cubic-bezier(.2,1.25,.3,1)}.rg-cur-g.in{transform:scale(1)}
.rg-cur{fill:url(#rg);stroke:var(--moss);stroke-width:2.2;filter:drop-shadow(0 0 8px rgba(63,207,142,.5))}.rg-dot{fill:var(--ember)}
.crit{display:grid;gap:8px}
.crit details{border:1px solid var(--line);border-radius:16px 5px 16px 5px;padding:10px 14px;background:rgba(10,14,12,.5)}
.crit summary{display:grid;grid-template-columns:96px 1fr 44px;gap:10px;align-items:center;cursor:pointer;list-style:none;font-size:14px}
.crit summary::-webkit-details-marker{display:none}
.crit .tr{height:8px;border-radius:8px;background:rgba(255,255,255,.07);overflow:hidden}.crit .fl{height:100%;width:0;border-radius:8px;transition:width 1s cubic-bezier(.2,.9,.2,1)}
.crit p{margin:10px 0 2px;font-size:13.5px;line-height:1.55;color:var(--mute)}
.lst{display:grid;gap:8px;margin:0;padding:0;list-style:none}
.lst li{padding:10px 14px;font-size:13.5px;line-height:1.5;border-radius:4px 16px 4px 16px;animation:rise .5s backwards}
.l-s li{background:rgba(31,122,69,.18);border-left:3px solid var(--moss)}.l-w li{background:rgba(220,20,60,.1);border-left:3px solid var(--crimson)}
.l-m li{border:1px dashed rgba(220,20,60,.55);background:transparent}.l-r li{background:rgba(255,255,255,.04);border-left:3px solid var(--forest-2)}
h3.sub3{font-size:13px;letter-spacing:.18em;text-transform:uppercase;color:var(--mute);margin:0 0 12px;font-weight:600}
pre.opt{margin:0;white-space:pre-wrap;word-break:break-word;font-family:'JetBrains Mono',monospace;font-size:13px;line-height:1.6;background:rgba(3,6,4,.8);border:1px solid var(--line);border-radius:18px 6px 18px 6px;padding:16px;min-height:90px;max-height:300px;overflow:auto}
pre.opt.typing::after{content:'\258D';color:var(--crimson);animation:blink 1s steps(2,start) infinite}
.battle{display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-top:14px}
.battle>div{padding:14px;border-radius:18px 6px 18px 6px;background:rgba(3,6,4,.7);border:1px solid var(--line);font-size:13px;line-height:1.55;white-space:pre-wrap;max-height:320px;overflow:auto}
.battle h5{margin:0 0 8px;font-size:11px;letter-spacing:.2em;text-transform:uppercase;font-weight:600}
#trend{width:100%;height:auto;display:block;overflow:visible}
.hist{width:100%;border-collapse:collapse;margin-top:12px;font-size:13px}
.hist td,.hist th{padding:8px 6px;border-bottom:1px solid var(--line);text-align:left}.hist th{font-size:11px;letter-spacing:.15em;text-transform:uppercase;color:var(--mute);font-weight:500}
.pillrow{display:flex;justify-content:center;gap:clamp(8px,2.4vw,26px);flex-wrap:nowrap;margin:44px 0 20px;perspective:700px}
.pill{width:clamp(62px,9.5vw,104px);display:flex;flex-direction:column;align-items:center;gap:10px;cursor:pointer;background:none;border:none;padding:0;
 transform:translate(var(--x,0px),var(--y,0px)) rotate(var(--r,0deg)) scale(var(--s,1));opacity:var(--o,1);transform-origin:center;will-change:transform}
.pill .ic{width:100%;aspect-ratio:1;border-radius:40% 60% 55% 45%/50% 42% 58% 50%;display:grid;place-items:center;background:radial-gradient(circle at 30% 25%,rgba(63,207,142,.25),rgba(14,18,16,.95) 70%);border:1px solid var(--line);transition:.4s;color:var(--moss)}
.pill svg{width:52%;fill:none;stroke:currentColor;stroke-width:2.2;stroke-linecap:round;stroke-linejoin:round}
.pill:hover .ic{border-radius:55% 45% 40% 60%/45% 58% 42% 55%;color:var(--ember);border-color:var(--crimson);box-shadow:0 0 calc(28px*var(--glow)) rgba(220,20,60,.45);transform:scale(1.1) rotate(-6deg)}
.pill span{font-size:11.5px;letter-spacing:.12em;text-transform:uppercase}
.pill-tip{text-align:center;color:var(--mute);font-size:13px;min-height:40px;max-width:520px;margin:0 auto}
.pill-h{font-size:clamp(26px,4.6vw,52px);text-align:center;color:var(--text)}
.reel-tile{position:relative;height:300px;margin:0 auto;max-width:760px;border-radius:44px 12px 44px 12px;overflow:hidden;cursor:none;border:1px solid var(--line)}
.reel-tile canvas{width:100%;height:100%;object-fit:cover;display:block}
.follow{position:absolute;left:0;top:0;z-index:3;display:flex;gap:8px;align-items:center;padding:8px;font-size:13px;color:#fff;mix-blend-mode:exclusion;pointer-events:none;user-select:none;letter-spacing:.08em;text-transform:uppercase}
#reelModal{position:absolute;inset:0;z-index:60;display:none;align-items:center;justify-content:center}#reelModal.on{display:flex}
.rm-back{position:absolute;inset:0;background:rgba(6,8,7,.88);backdrop-filter:blur(14px);opacity:0;transition:opacity .25s}
.rm-box{position:relative;width:min(94%,980px);aspect-ratio:16/9;border-radius:18px;overflow:hidden;background:#000;box-shadow:0 0 80px rgba(220,20,60,.25)}
.rm-box canvas{width:100%;height:100%;display:block}
.rm-x{position:absolute;right:10px;top:8px;z-index:3;background:none;border:none;font-size:30px;line-height:1;cursor:pointer;transform:rotate(45deg);color:#fff}
.rm-bar{position:absolute;left:0;right:0;bottom:0;display:flex;align-items:center;gap:12px;padding:14px 22px;background:linear-gradient(transparent,rgba(0,0,0,.7));mix-blend-mode:normal}
.rm-bar button{background:none;border:none;cursor:pointer;padding:4px;width:30px;height:30px;color:#fff}.rm-bar svg{width:18px;height:18px;fill:#fff}
.rm-bar span{font-family:'JetBrains Mono',monospace;font-size:11.5px;white-space:nowrap}
#rmRange{flex:1;-webkit-appearance:none;appearance:none;height:2px;background:rgba(255,255,255,.35);outline:none;border-radius:2px}
#rmRange::-webkit-slider-thumb{-webkit-appearance:none;width:0;height:0;border-radius:50%;background:var(--crimson);transition:.2s}#rmRange:hover::-webkit-slider-thumb{width:12px;height:12px}
.foot{padding:30px 34px 70px;text-align:center;font-size:12px;color:var(--mute);letter-spacing:.06em}
#dialBtn{position:absolute;right:18px;bottom:18px;z-index:40;width:44px;height:44px;border-radius:50%;border:1px solid var(--line);background:rgba(14,18,16,.9);cursor:pointer;font-size:18px;color:var(--moss);box-shadow:0 0 calc(18px*var(--glow)) rgba(63,207,142,.3)}
#dial{position:absolute;right:18px;bottom:74px;z-index:40;width:270px;padding:16px;display:none;border-radius:24px 8px 24px 8px;background:rgba(10,14,12,.96);border:1px solid var(--line);font-size:12px}
#dial.on{display:block;animation:rise .3s}
#dial h6{margin:0 0 10px;font-size:11px;letter-spacing:.22em;text-transform:uppercase;color:var(--moss);font-weight:600}
.dr{display:grid;grid-template-columns:96px 1fr 38px;gap:8px;align-items:center;margin:7px 0}.dr input{accent-color:var(--crimson);width:100%}.dr output{font-family:'JetBrains Mono',monospace;font-size:10.5px;text-align:right}
.dr select{background:#0e1210;border:1px solid var(--line);border-radius:8px;padding:4px;grid-column:2/4}
#toast{position:absolute;left:50%;bottom:22px;transform:translate(-50%,40px);opacity:0;z-index:80;padding:11px 20px;border-radius:16px 5px 16px 5px;background:rgba(14,18,16,.96);border:1px solid var(--moss);font-size:13px;transition:.4s;pointer-events:none;max-width:80%}
#toast.on{transform:translate(-50%,0);opacity:1}#toast.err{border-color:var(--crimson)}
</style>

<div id="ps">
<svg width="0" height="0" style="position:absolute"><defs><filter id="goo"><feGaussianBlur in="SourceGraphic" stdDeviation="3" result="b"/><feColorMatrix in="b" values="1 0 0 0 0  0 1 0 0 0  0 0 1 0 0  0 0 0 20 -8"/></filter></defs></svg>
<canvas id="psBg"></canvas>

<header class="top">
  <div class="logo"><svg viewBox="0 0 40 40"><path d="M20 3c9 0 17 7 17 17 0 11-9 17-17 17S3 31 3 20C3 10 11 3 20 3Z" fill="#0e1210" stroke="#3fcf8e" stroke-width="1.5"/><path d="M12 24c4-9 8 4 12-6 2-5 4-2 5 0" fill="none" stroke="#dc143c" stroke-width="2.2" stroke-linecap="round"/><circle cx="29" cy="18" r="2.4" fill="#ff3b5c"/></svg><span data-scramble>PROMPT·LAB</span></div>
  <nav class="nav"><button data-go="hero">Home</button><button data-go="studio">Studio</button><button data-go="pillars">Pillars</button><button data-go="reel">Reel</button></nav>
  <select id="model" title="Judge model"><option>openai/gpt-oss-20b</option><option>openai/gpt-oss-120b</option><option>llama-3.3-70b-versatile</option></select>
</header>

<main id="scroller">
<section id="hero">
  <div class="blob b1"></div><div class="blob b2"></div><div class="blob b3"></div>
  <div class="eyebrow" data-scramble>bio-electric prompt lab</div>
  <div class="kinetic hero-l1" data-text="Evolve your"></div>
  <div class="kinetic hero-l2" data-text="Prompt"></div>
  <p class="sub"><b>Write → Evaluate → Improve → Master.</b> Move your cursor over the letters. An AI judge scores six pillars of your prompt, then grows a stronger one.</p>
  <div class="hint">Scroll to enter the lab</div>
  <div class="legend"><span><i style="border:1.5px solid #3fcf8e"></i>direct follow</span><span><i style="background:#dc143c"></i>spring follow</span></div>
  <div class="dot" id="dot"></div><div class="orb" id="orb"></div>
</section>

<section id="studio">
  <span class="lab" data-scramble>01 · Select your challenge</span>
  <div class="card edge">
    <div class="row" id="domains"></div>
    <div class="row" style="margin-top:12px;justify-content:space-between"><div class="seg" id="levels"></div><button class="btn" id="shuffle">⟳ Surprise me</button></div>
    <div class="ch-card"><h4 id="chTitle"></h4><p id="chDesc"></p></div>
  </div>

  <div style="height:26px"></div>
  <span class="lab" data-scramble>02 · Write your prompt</span>
  <div class="grid2">
    <div class="card">
      <div class="caret-wrap"><textarea id="prompt" spellcheck="false" placeholder="Write your prompt here… (Ctrl + Enter to evaluate)"></textarea></div>
      <div class="stats"><span><b id="sW">0</b> words</span><span><b id="sC">0</b> chars</span><span>≈ <b id="sT">0</b> tokens</span></div>
      <div class="row">
        <button class="btn pri" id="evalBtn">⚡ Evaluate prompt</button>
        <button class="btn" id="scaffold">▦ Six-part scaffold</button>
        <button class="btn" id="clear">Clear</button>
      </div>
    </div>
    <div class="card">
      <h3 class="sub3">Live quick-scan</h3>
      <div class="qs" id="qs"></div>
      <p class="note">Instant keyword heuristics while you type. The real score comes from the AI judge.</p>
      <h3 class="sub3" style="margin-top:20px">Session label</h3>
      <div class="caret-wrap"><input id="label" placeholder="e.g. v1 — add a role" autocomplete="off"></div>
    </div>
  </div>

  <div id="results" style="margin-top:34px">
    <span class="lab" data-scramble>03 · Evaluation dashboard</span>
    <div class="card edge">
      <div class="hdrstats">
        <div class="gauge-w"><svg class="gauge" viewBox="0 0 140 140"><circle class="g-track" cx="70" cy="70" r="54"/><circle id="gArc" class="g-arc" cx="70" cy="70" r="54" transform="rotate(-90 70 70)"/></svg><div class="g-num"><b id="score">0</b><small>/100</small></div></div>
        <div><div class="rate" id="rating" data-scramble>—</div><div class="kpis"><div class="kpi">Change<b id="kDelta">—</b></div><div class="kpi">Attempts<b id="kAtt">0</b></div><div class="kpi">Best<b id="kBest">0</b></div></div><div class="dchips" id="dchips"></div></div>
      </div>
    </div>
    <div class="grid3">
      <div class="card"><h3 class="sub3">Prompt quality radar <span style="float:right;text-transform:none;letter-spacing:0;color:var(--crimson)">- - previous</span></h3><svg id="radar" viewBox="-30 -10 460 420"></svg></div>
      <div class="card"><h3 class="sub3">Detailed feedback</h3><div class="crit" id="crit"></div></div>
    </div>
    <div class="grid3">
      <div class="card"><h3 class="sub3">Strengths</h3><ul class="lst l-s" id="ul-s"></ul><h3 class="sub3" style="margin-top:18px">Weaknesses</h3><ul class="lst l-w" id="ul-w"></ul></div>
      <div class="card"><h3 class="sub3">Missing components</h3><ul class="lst l-m" id="ul-m"></ul><h3 class="sub3" style="margin-top:18px">Recommendations</h3><ul class="lst l-r" id="ul-r"></ul></div>
    </div>
    <div class="card edge" style="margin-top:18px">
      <h3 class="sub3">Optimized prompt</h3>
      <pre class="opt" id="opt"></pre>
      <div class="row" style="margin-top:14px"><button class="btn" id="copy">Copy</button><button class="btn" id="use">↳ Use in editor</button><button class="btn pri" id="useEval">⚡ Use &amp; re-evaluate</button><button class="btn" id="battleBtn">⚔ Battle test</button></div>
      <div class="battle" id="battle" style="display:none"><div><h5 style="color:var(--ember)">Your prompt → output</h5><span id="bA"></span></div><div><h5 style="color:var(--moss)">Optimized → output</h5><span id="bB"></span></div></div>
    </div>
    <div class="card" style="margin-top:18px">
      <h3 class="sub3">Prompt improvement journey</h3>
      <svg id="trend" viewBox="0 0 640 220"></svg>
      <table class="hist"><thead><tr><th>#</th><th>Label</th><th>Score</th><th>Δ</th><th></th></tr></thead><tbody id="hist"></tbody></table>
      <div class="row" style="margin-top:14px"><button class="btn" id="exMd">⇩ Report .md</button><button class="btn" id="exJson">⇩ History .json</button></div>
    </div>
  </div>
</section>

<section id="pillars" style="padding-top:90px">
  <div class="kinetic pill-h" data-text="Six pillars of a great prompt"></div>
  <div class="pillrow" id="pillrow"></div>
  <p class="pill-tip" id="pillTip">Hover a pillar to read it. Click one to drop a ready-made snippet into your editor.</p>
</section>

<section id="reel" style="padding-top:70px">
  <div style="text-align:center"><span class="lab" data-scramble>The 32-second flow</span><h2 class="sec">Watch the lab work</h2></div>
  <div class="reel-tile" id="reelTile"><div class="follow" id="follow">▶ Play</div><canvas id="reelPrev" width="640" height="360"></canvas></div>
</section>

<div class="foot">Python · Groq · LLM-as-a-Judge · hand-rolled SVG charts<br>Motion ideas inspired by Skiper UI (gxuri)</div>
</main>

<button id="dialBtn" title="Dial controls">◉</button>
<aside id="dial"><h6>Dial · motion lab</h6><div id="dialRows"></div></aside>

<div id="reelModal"><div class="rm-back" id="rmBack"></div>
  <div class="rm-box" id="reelBox"><canvas id="reelCv" width="960" height="540"></canvas><button class="rm-x" id="rmX" aria-label="Close">+</button>
    <div class="rm-bar"><button id="rmPlay"></button><span id="rmTime">0:00 / 0:32</span><input type="range" id="rmRange" min="0" max="3199" value="0"><button id="rmMute"></button></div></div></div>
<div id="toast"></div>
</div>

<script>
(()=>{
const $=(s,r=document)=>r.querySelector(s),$$=(s,r=document)=>[...r.querySelectorAll(s)];
const clamp=(v,a=0,b=1)=>Math.min(b,Math.max(a,v));
const el=(t,c,x)=>{const e=document.createElement(t);if(c)e.className=c;if(x!=null)e.textContent=x;return e};
const SCENARIOS=__SCENARIOS_JSON__, CRITERIA=__CRITERIA_JSON__;
const KEYS=Object.keys(CRITERIA);
const root=$('#ps'),sc=$('#scroller');
let RM=matchMedia('(prefers-reduced-motion: reduce)').matches;
const P={caret:{k:500,c:30,m:.5},orb:{k:131,c:10,m:.1},fs:17,glow:1};

/* ---------- spring physics ---------- */
class Spring{constructor(v,o){this.x=v;this.v=0;this.t=v;this.o=o}
 step(dt){if(RM){this.x=this.t;this.v=0;return this.x}
  const n=Math.max(1,Math.ceil(dt*240)),h=dt/n,{k,c,m}=this.o;
  for(let i=0;i<n;i++){this.v+=(-k*(this.x-this.t)-c*this.v)/m*h;this.x+=this.v*h}return this.x}
 get settled(){return Math.abs(this.v)<.01&&Math.abs(this.x-this.t)<.01}}
const updaters=[];let lastT=performance.now();
(function loop(now){const dt=Math.min(.05,(now-lastT)/1000);lastT=now;for(const u of updaters)u(dt,now);requestAnimationFrame(loop)})(performance.now());

/* ---------- python bridge (+ offline mock so the UI can be previewed) ---------- */
const IN_COLAB=!!(window.google&&google.colab&&google.colab.kernel);
const STANDALONE=!IN_COLAB&&/^https?:/.test(location.protocol);
if(STANDALONE){root.style.height='100vh';root.style.borderRadius='0';document.documentElement.style.cssText=document.body.style.cssText='margin:0;overflow:hidden;background:#050706'}
async function py(name,args){
 if(IN_COLAB){const r=await google.colab.kernel.invokeFunction(name,args,{});return r.data['application/json']}
 if(STANDALONE){const r=await fetch('api/'+name,{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({args})});return await r.json()}
 return mock(name,args)}
function mock(name,a){return new Promise(res=>setTimeout(()=>{
 if(name==='ps_evaluate'){const s=scan(a[0]),sc={},ex={};KEYS.forEach(k=>{sc[k]=Math.min(10,Math.round(2+s[k]*6+Math.random()*2));ex[k]='Demo mode (not connected to Colab): heuristic estimate for '+k+'.'});
  return res({scores:sc,explanations:ex,strengths:['Clear intent'],weaknesses:['Demo weakness'],missing_components:KEYS.filter(k=>s[k]<.5),recommendations:['Add more detail'],optimized_prompt:'You are an expert.\n'+a[0]+'\nFormat: numbered list.\nTone: professional.',overall_score:Math.round(Object.values(sc).reduce((x,y)=>x+y,0)/60*100)})}
 if(name==='ps_battle')return res({a:'(demo output A)',b:'(demo output B)'});
 res({path:'(demo mode)'})},700))}

/* ---------- toast / clipboard ---------- */
let toastT;function toast(m,err){const t=$('#toast');t.textContent=m;t.className='on'+(err?' err':'');clearTimeout(toastT);toastT=setTimeout(()=>t.className='',err?5200:2600)}
function copyText(txt){try{navigator.clipboard.writeText(txt).then(()=>toast('Copied'),fb)}catch(e){fb()}
 function fb(){const x=el('textarea');x.value=txt;x.style.cssText='position:fixed;opacity:0';document.body.appendChild(x);x.select();try{document.execCommand('copy');toast('Copied')}catch(e){toast('Copy blocked by browser',1)}x.remove()}}

/* ---------- electro-mycelium background ---------- */
const bg=$('#psBg'),bx=bg.getContext('2d');let BW=0,BH=0,nodes=[];const ptr={x:-999,y:-999,on:false};
function sizeBg(){const r=root.getBoundingClientRect(),d=Math.min(2,devicePixelRatio||1);BW=r.width;BH=r.height;bg.width=BW*d;bg.height=BH*d;bx.setTransform(d,0,0,d,0,0);
 if(!nodes.length)nodes=Array.from({length:62},()=>({x:Math.random()*BW,y:Math.random()*BH,vx:(Math.random()-.5)*.28,vy:(Math.random()-.5)*.28,r:1+Math.random()*2.2}))}
new ResizeObserver(sizeBg).observe(root);sizeBg();
root.addEventListener('pointermove',e=>{const r=root.getBoundingClientRect();ptr.x=e.clientX-r.left;ptr.y=e.clientY-r.top;ptr.on=true});
root.addEventListener('pointerleave',()=>ptr.on=false);
updaters.push((dt,now)=>{bx.clearRect(0,0,BW,BH);const g=P.glow;
 for(const n of nodes){n.x+=n.vx;n.y+=n.vy;if(n.x<0||n.x>BW)n.vx*=-1;if(n.y<0||n.y>BH)n.vy*=-1}
 bx.lineWidth=1;
 for(let i=0;i<nodes.length;i++)for(let j=i+1;j<nodes.length;j++){const a=nodes[i],b=nodes[j],d=Math.hypot(a.x-b.x,a.y-b.y);
  if(d<135){bx.strokeStyle=`rgba(63,207,142,${(1-d/135)*.22*g})`;bx.beginPath();bx.moveTo(a.x,a.y);bx.lineTo(b.x,b.y);bx.stroke()}}
 for(const n of nodes){bx.fillStyle='rgba(63,207,142,'+.5*g+')';bx.beginPath();bx.arc(n.x,n.y,n.r,0,6.283);bx.fill()}
 if(ptr.on&&!RM){const near=nodes.map(n=>[n,Math.hypot(n.x-ptr.x,n.y-ptr.y)]).filter(x=>x[1]<190).sort((a,b)=>a[1]-b[1]).slice(0,5);
  for(const [n,d] of near){bx.strokeStyle=`rgba(255,59,92,${(1-d/190)*.85*g})`;bx.lineWidth=1.3;bx.beginPath();bx.moveTo(ptr.x,ptr.y);
   for(let s=1;s<=5;s++){const f=s/5,jx=(Math.random()-.5)*(s<5?14:0),jy=(Math.random()-.5)*(s<5?14:0);bx.lineTo(ptr.x+(n.x-ptr.x)*f+jx,ptr.y+(n.y-ptr.y)*f+jy)}bx.stroke()}}
});

/* ---------- interactive typography ---------- */
function split(e){const txt=e.dataset.text||e.textContent.trim();e.textContent='';e.setAttribute('aria-label',txt);
 const cs=[...txt].map(c=>{const s=el('span','ch',c===' '?'\u00a0':c);s.setAttribute('aria-hidden','true');e.appendChild(s);return s});
 const mid=Math.floor(cs.length/2);cs.forEach((s,i)=>s.dataset.d=i-mid);e.chars=cs}
const kin=$$('.kinetic');kin.forEach(split);
let kPend=false,kx=0,ky=0,kOn=false;
function applyK(){kPend=false;for(const e of kin)for(const s of e.chars){if(!kOn){s.style.setProperty('--k',0);continue}
 const r=s.getBoundingClientRect(),d=Math.hypot(kx-(r.left+r.width/2),ky-(r.top+r.height/2));s.style.setProperty('--k',Math.exp(-(d*d)/(2*95*95)).toFixed(3))}}
root.addEventListener('pointermove',e=>{kx=e.clientX;ky=e.clientY;kOn=true;if(!kPend){kPend=true;requestAnimationFrame(applyK)}});
root.addEventListener('pointerleave',()=>{kOn=false;applyK()});
function scramble(e){const txt=e.dataset.t||(e.dataset.t=e.textContent),G='▓▒░<>/\\|{}[]#%&*+=ϟ';let f=0;clearInterval(e._i);
 e._i=setInterval(()=>{f++;e.textContent=[...txt].map((c,i)=>c===' '?' ':(i<f/1.4?c:G[Math.random()*G.length|0])).join('');if(f>txt.length*1.4+2){clearInterval(e._i);e.textContent=txt}},26)}
$$('[data-scramble]').forEach(e=>e.addEventListener('pointerenter',()=>scramble(e)));

/* scroll-driven letters + icons (Skiper-31 style) */
const prog=e=>{const a=sc.getBoundingClientRect(),r=e.getBoundingClientRect();return clamp((sc.clientHeight-(r.top-a.top))/(sc.clientHeight+r.height))};
const ease=t=>1-Math.pow(1-t,3);
const intro=new Spring(0,{k:55,c:11,m:1});intro.t=1;
const heroChars=[...$('.hero-l1').chars,...$('.hero-l2').chars],pillH=$('.pill-h');
let sDirty=true;sc.addEventListener('scroll',()=>sDirty=true,{passive:true});
function buildPillars(){
 const IC={Role:'<circle cx="24" cy="16" r="7"/><path d="M8 42c2-10 9-14 16-14s14 4 16 14"/>',Goal:'<circle cx="24" cy="24" r="16"/><circle cx="24" cy="24" r="9"/><circle cx="24" cy="24" r="2.5"/>',
 Context:'<path d="M24 6 42 15 24 24 6 15z"/><path d="M6 24l18 9 18-9"/><path d="M6 33l18 9 18-9"/>',Constraints:'<path d="M24 5 40 11v12c0 10-7 17-16 20C15 40 8 33 8 23V11z"/><path d="M17 24l5 5 9-10"/>',
 Format:'<rect x="7" y="8" width="34" height="32" rx="4"/><path d="M7 18h34M18 18v22"/>',Style:'<path d="M8 40C14 22 26 10 40 8c-1 14-12 26-30 32z"/><path d="M8 40 24 24"/>'};
 const SN={Role:'Role: You are an expert [role / persona].',Goal:'Goal: Your task is to [what you need done].',Context:'Context: [background, audience, data, situation].',Constraints:'Constraints: [rules, limits, things to avoid].',Format:'Format: [structure of the output, e.g. numbered list, table, JSON].',Style:'Style: [tone, voice, reading level].'};
 const row=$('#pillrow');KEYS.forEach(k=>{const b=el('button','pill');b.innerHTML='<div class="ic"><svg viewBox="0 0 48 48">'+(IC[k]||'')+'</svg></div><span>'+k+'</span>';
  b.addEventListener('pointerenter',()=>$('#pillTip').textContent=CRITERIA[k]);
  b.addEventListener('click',()=>{insertSnippet(SN[k]);sc.scrollTo({top:$('#studio').offsetTop+150});toast(k+' snippet added')});row.appendChild(b)})}
buildPillars();
const pills=$$('.pill');
updaters.push(dt=>{intro.step(dt);const animating=!intro.settled;if(!sDirty&&!animating)return;sDirty=false;
 const q=clamp(sc.scrollTop/520);
 for(const s of heroChars){const d=+s.dataset.d;s.style.setProperty('--x',(d*55*(1-intro.x)+d*30*q)+'px');s.style.setProperty('--rx',(d*45*(1-intro.x)+d*25*q)+'deg');s.style.setProperty('--o',clamp(intro.x*1.2)*(1-q*.85))}
 const e1=ease(clamp(prog(pillH)/.5));for(const s of pillH.chars){const d=+s.dataset.d;s.style.setProperty('--x',d*14*(1-e1)+'px');s.style.setProperty('--rx',d*50*(1-e1)+'deg');s.style.setProperty('--o',.1+.9*e1)}
 const e2=ease(clamp(prog($('#pillrow'))/.5));pills.forEach((p,i)=>{const d=i-(pills.length-1)/2;p.style.setProperty('--x',d*90*(1-e2)+'px');p.style.setProperty('--r',d*50*(1-e2)+'deg');p.style.setProperty('--y',-Math.abs(d)*20*(1-e2)+'px');p.style.setProperty('--s',.75+.25*e2);p.style.setProperty('--o',.2+.8*e2)})});

/* ---------- mouse-follow: direct dot vs spring orb ---------- */
const hero=$('#hero'),dot=$('#dot'),orb=$('#orb');
const oX=new Spring(0,P.orb),oY=new Spring(0,P.orb),oO=new Spring(0,P.orb),oS=new Spring(0,P.orb);
hero.addEventListener('pointermove',e=>{const b=hero.getBoundingClientRect(),x=e.clientX-b.left,y=e.clientY-b.top;dot.style.transform=`translate(${x}px,${y}px)`;oX.t=x;oY.t=y});
hero.addEventListener('pointerenter',e=>{const b=hero.getBoundingClientRect();if(oO.x<.05){oX.x=oX.t=e.clientX-b.left;oY.x=oY.t=e.clientY-b.top}oO.t=1;oS.t=1;dot.style.opacity=1});
hero.addEventListener('pointerleave',()=>{oO.t=0;oS.t=0;dot.style.opacity=0});
updaters.push(dt=>{if(oO.settled&&oS.settled&&oX.settled&&oY.settled)return;oX.step(dt);oY.step(dt);oO.step(dt);oS.step(dt);orb.style.opacity=clamp(oO.x);orb.style.transform=`translate(${oX.x}px,${oY.x}px) scale(${Math.max(0,oS.x)})`});

/* ---------- smooth spring caret ---------- */
const carets=[];
class SmoothCaret{
 constructor(inp){this.el=inp;this.wrap=inp.parentElement;this.sx=new Spring(0,P.caret);this.sy=new Spring(0,P.caret);this.shown=false;
  this.c=el('div','scaret');this.wrap.appendChild(this.c);this.m=el('div','mirror');this.wrap.appendChild(this.m);
  const u=()=>requestAnimationFrame(()=>this.update());
  ['input','keyup','click','focus','scroll','select','mouseup'].forEach(ev=>inp.addEventListener(ev,u));
  inp.addEventListener('input',()=>{this.c.classList.remove('idle');clearTimeout(this.ti);this.ti=setTimeout(()=>this.c.classList.add('idle'),650)});
  inp.addEventListener('focus',()=>this.c.classList.add('idle'));
  inp.addEventListener('blur',()=>{this.shown=false;this.c.classList.remove('show','idle')});
  document.addEventListener('selectionchange',()=>{if(document.activeElement===inp)u()});
  new ResizeObserver(()=>this.update()).observe(inp);
  updaters.push(dt=>this.frame(dt));carets.push(this)}
 update(){const e=this.el;if(document.activeElement!==e)return;const cs=getComputedStyle(e),m=this.m,ta=e.tagName==='TEXTAREA';
  ['fontFamily','fontSize','fontWeight','fontStyle','letterSpacing','lineHeight','textTransform','textIndent','wordSpacing','paddingTop','paddingRight','paddingBottom','paddingLeft','borderTopWidth','borderRightWidth','borderBottomWidth','borderLeftWidth','tabSize'].forEach(k=>m.style[k]=cs[k]);
  m.style.boxSizing='border-box';m.style.borderStyle='solid';m.style.borderColor='transparent';m.style.whiteSpace=ta?'pre-wrap':'pre';
  const bl=parseFloat(cs.borderLeftWidth)||0,bt=parseFloat(cs.borderTopWidth)||0;m.style.width=(e.clientWidth+bl+(parseFloat(cs.borderRightWidth)||0))+'px';
  const s=e.selectionStart??0,en=e.selectionEnd??0,idx=s===en?s:(e.selectionDirection==='backward'?s:en);
  m.textContent=e.type==='password'?'•'.repeat(idx):e.value.slice(0,idx);const sp=el('span','','\u200b');m.appendChild(sp);
  const fs=parseFloat(cs.fontSize),ch=fs*1.15,lh=parseFloat(cs.lineHeight)||fs*1.4;
  const x=sp.offsetLeft+bl-e.scrollLeft;let y=ta?sp.offsetTop+bt-e.scrollTop+(lh-ch)/2:(e.offsetHeight-ch)/2;
  const pl=parseFloat(cs.paddingLeft)||0,pr=parseFloat(cs.paddingRight)||0,vis=x>=pl-2&&x<=e.clientWidth-pr+2&&y>=-2&&y<=e.clientHeight-ch+4;
  this.c.style.height=ch+'px';this.sx.t=x;this.sy.t=y;
  if(!this.shown){this.sx.x=x;this.sy.x=y;this.sx.v=this.sy.v=0;this.shown=true;this.frame(0)}
  this.c.classList.toggle('show',vis&&s===en)}
 frame(dt){if(!this.shown)return;if(dt)for(const s of [this.sx,this.sy])s.step(dt);this.c.style.transform=`translate(${this.sx.x}px,${this.sy.x}px)`}}
const ta=$('#prompt'),lab=$('#label');new SmoothCaret(ta);new SmoothCaret(lab);
function insertSnippet(t){const v=ta.value;const pre=v&&!v.endsWith('\n')?'\n':'';ta.setRangeText(pre+t+'\n',ta.value.length,ta.value.length,'end');ta.dispatchEvent(new Event('input'));ta.focus()}

/* ---------- challenge picker ---------- */
const state={domain:'HR',level:'Intermediate',attempts:[],last:null};
const LEVELS=['Beginner','Intermediate','Advanced'];
function renderChallenge(){$$('#domains .chip').forEach(c=>c.classList.toggle('on',c.dataset.v===state.domain));$$('#levels button').forEach(c=>c.classList.toggle('on',c.dataset.v===state.level));
 const c=SCENARIOS[state.domain][state.level];$('#chTitle').textContent=c.title;$('#chDesc').textContent=c.description.trim().replace(/\s+/g,' ')}
Object.keys(SCENARIOS).forEach(d=>{const b=el('button','chip',d);b.dataset.v=d;b.onclick=()=>{state.domain=d;renderChallenge()};$('#domains').appendChild(b)});
LEVELS.forEach(l=>{const b=el('button','',l);b.dataset.v=l;b.onclick=()=>{state.level=l;renderChallenge()};$('#levels').appendChild(b)});
$('#shuffle').onclick=()=>{const ds=Object.keys(SCENARIOS);state.domain=ds[Math.random()*ds.length|0];state.level=LEVELS[Math.random()*3|0];renderChallenge()};
renderChallenge();

/* ---------- live quick-scan ---------- */
const RX={Role:/\b(you are|act as|acting as|your role|expert|specialist|persona|as an? [a-z]+ (manager|engineer|analyst|developer|writer|consultant|recruiter))\b/gi,
 Goal:/\b(create|write|generate|analy[sz]e|summari[sz]e|explain|design|build|evaluate|convert|produce|draft|identify|compare|plan|your task|goal|objective)\b/gi,
 Context:/\b(context|background|audience|target|scenario|situation|given|because|company|client|customer|team|dataset|data|for an?|about)\b/gi,
 Constraints:/\b(must|do not|don't|never|avoid|only|limit|at most|at least|no more than|within|under|exactly|between|maximum|minimum|\d+\s*(words|sentences|items|bullets|questions|points|paragraphs))\b/gi,
 Format:/\b(format|bullet|numbered|table|json|markdown|heading|section|list|steps|paragraph|outline|schema|columns?|structure)\b/gi,
 Style:/\b(tone|style|voice|professional|friendly|formal|concise|casual|persuasive|simple|technical|witty|empathetic|clear|engaging)\b/gi};
function scan(t){const o={};for(const k of KEYS){const m=t.match(RX[k]);o[k]=clamp((m?new Set(m.map(x=>x.toLowerCase())).size:0)/2)}
 if(t.split(/\s+/).filter(Boolean).length>40)o.Context=Math.max(o.Context,.5);return o}
KEYS.forEach(k=>{const r=el('div','qb');r.id='qb-'+k;r.innerHTML='<span>'+k+'</span><div class="tr"><div class="fl"></div></div><b class="ok">✓</b>';r.style.setProperty('--v',0);$('#qs').appendChild(r)});
function liveScan(){const t=ta.value,s=scan(t),w=t.split(/\s+/).filter(Boolean).length;$('#sW').textContent=w;$('#sC').textContent=t.length;$('#sT').textContent=Math.ceil(t.length/4);
 KEYS.forEach(k=>{const r=$('#qb-'+k);r.style.setProperty('--v',s[k]);r.classList.toggle('full',s[k]>=1)})}
ta.addEventListener('input',liveScan);liveScan();
$('#scaffold').onclick=()=>{ta.value=['Role: You are an expert [role].','Goal: [what you need done].','Context: [background, audience, data].','Constraints: [rules, limits, things to avoid].','Format: [structure of the output].','Style: [tone, voice].'].join('\n');ta.dispatchEvent(new Event('input'));ta.focus()};
$('#clear').onclick=()=>{ta.value='';ta.dispatchEvent(new Event('input'));ta.focus()};
ta.addEventListener('keydown',e=>{if((e.ctrlKey||e.metaKey)&&e.key==='Enter'){e.preventDefault();evaluate()}});

/* ---------- evaluation ---------- */
const mix=t=>{const a=[220,20,60],b=[47,168,104],c=a.map((v,i)=>Math.round(v+(b[i]-v)*clamp(t)));return `rgb(${c})`};
const rate=s=>s>=90?'Excellent':s>=75?'Strong':s>=60?'Good':s>=40?'Needs work':'Beginner';
const fmt=v=>Number(Number(v).toFixed(1));
const esc=t=>String(t).replace(/[&<>"]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;'}[c]));
function countTo(node,from,to,ms=1100){const t0=performance.now();(function f(n){const p=clamp((n-t0)/ms);node.textContent=Math.round(from+(to-from)*ease(p));if(p<1)requestAnimationFrame(f)})(t0)}
function typewrite(node,text){clearInterval(node._t);node.textContent='';node.classList.add('typing');let i=0;const st=Math.max(2,Math.ceil(text.length/200));
 node._t=setInterval(()=>{i+=st;node.textContent=text.slice(0,i);if(i>=text.length){clearInterval(node._t);node.textContent=text;node.classList.remove('typing')}},16)}
function fillList(id,arr,empty){const u=$(id);u.innerHTML='';(arr.length?arr:[empty]).forEach((t,i)=>{const l=el('li','',t);l.style.animationDelay=i*70+'ms';u.appendChild(l)})}
async function evaluate(){
 const prompt=ta.value.trim();if(!prompt){toast('Write a prompt first',1);ta.classList.add('shake');setTimeout(()=>ta.classList.remove('shake'),450);return}
 const b=$('#evalBtn');b.disabled=true;b.classList.add('busy');b.textContent='Judge is thinking…';
 try{const c=SCENARIOS[state.domain][state.level];
  const res=await py('ps_evaluate',[prompt,state.domain,state.level,c.description.trim().replace(/\s+/g,' '),$('#model').value]);
  if(res.error)throw new Error(res.error);
  const prev=state.attempts[state.attempts.length-1];
  const rec={n:state.attempts.length+1,score:res.overall_score,scores:res.scores,prompt,optimized:res.optimized_prompt,domain:state.domain,level:state.level,model:$('#model').value,label:$('#label').value.trim()||('Attempt '+(state.attempts.length+1)),res};
  state.attempts.push(rec);state.last=rec;render(rec,prev)}
 catch(e){toast('⚠ '+(e&&e.message||JSON.stringify(e)),1)}
 finally{b.disabled=false;b.classList.remove('busy');b.textContent='⚡ Evaluate prompt'}}
function render(rec,prev){const r=rec.res,R=$('#results');R.classList.add('on');
 const gA=$('#gArc'),C=2*Math.PI*54;gA.style.strokeDasharray=C;gA.style.strokeDashoffset=C;gA.style.stroke=gA.style.color=mix(rec.score/100);requestAnimationFrame(()=>requestAnimationFrame(()=>gA.style.strokeDashoffset=C*(1-rec.score/100)));
 countTo($('#score'),+$('#score').textContent||0,rec.score);const rt=$('#rating');rt.dataset.t=rate(rec.score);rt.style.color=mix(rec.score/100);scramble(rt);
 const d=prev?rec.score-prev.score:null,kd=$('#kDelta');kd.textContent=d==null?'—':(d>0?'▲ +':d<0?'▼ ':'= ')+d;kd.className=d>0?'up':d<0?'down':'';
 $('#kAtt').textContent=state.attempts.length;$('#kBest').textContent=Math.max(...state.attempts.map(a=>a.score));
 const dc=$('#dchips');dc.innerHTML='';if(prev)KEYS.forEach(k=>{const x=rec.scores[k]-prev.scores[k],s=el('span','',k+' '+(x>0?'+':'')+fmt(x));s.style.color=x>0?'var(--moss)':x<0?'var(--ember)':'var(--mute)';dc.appendChild(s)});
 radar(rec.scores,prev&&prev.scores);
 const cr=$('#crit');cr.innerHTML='';KEYS.forEach(k=>{const v=rec.scores[k],dt=el('details');const sm=el('summary');sm.innerHTML='<b>'+k+'</b><div class="tr"><div class="fl"></div></div><span>'+fmt(v)+'/10</span>';dt.appendChild(sm);dt.appendChild(el('p','',r.explanations[k]||''));cr.appendChild(dt);
  const f=$('.fl',sm);f.style.background=mix(v/10);setTimeout(()=>f.style.width=v*10+'%',60)});
 fillList('#ul-s',r.strengths,'—');fillList('#ul-w',r.weaknesses,'—');fillList('#ul-m',r.missing_components,'All six components are present!');fillList('#ul-r',r.recommendations,'—');
 typewrite($('#opt'),r.optimized_prompt||'');$('#battle').style.display='none';trend();
 setTimeout(()=>sc.scrollTo({top:$('#results').offsetTop+$('#studio').offsetTop-70}),150)}
function radar(cur,prev){const C=200,R=140,n=KEYS.length,pt=(i,v)=>{const a=-Math.PI/2+i*2*Math.PI/n;return[C+Math.cos(a)*R*v/10,C+Math.sin(a)*R*v/10]};
 const poly=o=>KEYS.map((k,i)=>pt(i,o[k]).map(x=>x.toFixed(1)).join(',')).join(' ');
 let s='<defs><radialGradient id="rg"><stop offset="0" stop-color="#3fcf8e" stop-opacity=".2"/><stop offset="1" stop-color="#dc143c" stop-opacity=".5"/></radialGradient></defs>';
 for(const r of [2,4,6,8,10])s+='<polygon class="rg-ring" points="'+KEYS.map((_,i)=>pt(i,r).join(',')).join(' ')+'"/>';
 KEYS.forEach((k,i)=>{const[x,y]=pt(i,10),[lx,ly]=pt(i,12.3);s+=`<line class="rg-ax" x1="${C}" y1="${C}" x2="${x}" y2="${y}"/><text class="rg-lb" x="${lx}" y="${ly}" text-anchor="${lx<C-6?'end':lx>C+6?'start':'middle'}" dominant-baseline="middle">${k}</text>`});
 if(prev)s+='<polygon class="rg-prev" points="'+poly(prev)+'"/>';
 s+='<g class="rg-cur-g" id="rgCur"><polygon class="rg-cur" points="'+poly(cur)+'"/>'+KEYS.map((k,i)=>{const[x,y]=pt(i,cur[k]);return`<circle class="rg-dot" cx="${x}" cy="${y}" r="4"/>`}).join('')+'</g>';
 $('#radar').innerHTML=s;requestAnimationFrame(()=>requestAnimationFrame(()=>$('#rgCur').classList.add('in')))}
function trend(){const A=state.attempts,W=640,H=220,pd=34,x=i=>A.length===1?W/2:pd+i*(W-2*pd)/(A.length-1),y=v=>H-pd-v/100*(H-2*pd);
 let s='<defs><linearGradient id="tg" x1="0" x2="1"><stop offset="0" stop-color="#dc143c"/><stop offset="1" stop-color="#3fcf8e"/></linearGradient></defs>';
 [0,25,50,75,100].forEach(v=>s+=`<line x1="${pd}" x2="${W-pd}" y1="${y(v)}" y2="${y(v)}" stroke="rgba(120,190,150,.14)"/><text x="${pd-8}" y="${y(v)+4}" fill="#7e9388" font-size="10" text-anchor="end">${v}</text>`);
 if(A.length>1)s+='<polyline fill="none" stroke="url(#tg)" stroke-width="3" stroke-linecap="round" stroke-linejoin="round" points="'+A.map((a,i)=>x(i)+','+y(a.score)).join(' ')+'"/>';
 A.forEach((a,i)=>s+=`<g style="cursor:pointer" data-i="${i}"><circle cx="${x(i)}" cy="${y(a.score)}" r="6.5" fill="${mix(a.score/100)}" stroke="#090b0a" stroke-width="2"/><text x="${x(i)}" y="${y(a.score)-12}" fill="#e4eee8" font-size="11" text-anchor="middle">${a.score}</text><title>${esc(a.label)}: restore this prompt</title></g>`);
 $('#trend').innerHTML=s;$$('#trend g[data-i]').forEach(g=>g.onclick=()=>restore(+g.dataset.i));
 const tb=$('#hist');tb.innerHTML='';A.forEach((a,i)=>{const p=A[i-1],d=p?a.score-p.score:null,tr=el('tr');
  tr.innerHTML=`<td>${a.n}</td><td></td><td style="color:${mix(a.score/100)};font-weight:700">${a.score}</td><td class="${d>0?'up':d<0?'down':''}">${d==null?'—':(d>0?'+':'')+d}</td><td><button class="btn" style="padding:5px 12px">Restore</button></td>`;
  tr.children[1].textContent=a.label;$('button',tr).onclick=()=>restore(i);tb.appendChild(tr)})}
function restore(i){ta.value=state.attempts[i].prompt;ta.dispatchEvent(new Event('input'));sc.scrollTo({top:$('#studio').offsetTop+120});ta.focus();toast('Restored attempt '+(i+1))}
$('#evalBtn').onclick=evaluate;
$('#copy').onclick=()=>copyText(state.last?state.last.optimized:'');
$('#use').onclick=()=>{if(!state.last)return;ta.value=state.last.optimized;ta.dispatchEvent(new Event('input'));sc.scrollTo({top:$('#studio').offsetTop+120});ta.focus()};
$('#useEval').onclick=()=>{if(!state.last)return;ta.value=state.last.optimized;ta.dispatchEvent(new Event('input'));evaluate()};
$('#battleBtn').onclick=async()=>{if(!state.last)return;const b=$('#battleBtn');b.disabled=true;b.textContent='Running both…';
 try{const r=await py('ps_battle',[state.last.prompt,state.last.optimized,$('#model').value]);if(r.error)throw new Error(r.error);$('#battle').style.display='grid';$('#bA').textContent=r.a;$('#bB').textContent=r.b}
 catch(e){toast('⚠ '+(e&&e.message||JSON.stringify(e)),1)}finally{b.disabled=false;b.textContent='⚔ Battle test'}};

/* ---------- export ---------- */
function download(name,text,mime){try{const a=el('a');a.href=URL.createObjectURL(new Blob([text],{type:mime}));a.download=name;document.body.appendChild(a);a.click();a.remove()}catch(e){}
 py('ps_save',[name,text]).then(r=>toast('Saved to '+r.path)).catch(()=>{})}
$('#exJson').onclick=()=>{if(!state.attempts.length)return toast('Nothing to export yet',1);download('prompt_history.json',JSON.stringify(state.attempts.map(({res,...a})=>({...a,evaluation:res})),null,2),'application/json')};
$('#exMd').onclick=()=>{if(!state.attempts.length)return toast('Nothing to export yet',1);const A=state.attempts,L=state.last;
 let m='# Prompt Evaluation Report\n\n**Challenge:** '+L.domain+' / '+L.level+'\n\n## Journey\n\n| # | Label | Score |\n|---|---|---|\n'+A.map(a=>`| ${a.n} | ${a.label} | ${a.score} |`).join('\n')+'\n\n## Latest attempt\n\n```text\n'+L.prompt+'\n```\n\n### Scores\n\n'+KEYS.map(k=>`- **${k}**: ${fmt(L.scores[k])}/10 — ${L.res.explanations[k]}`).join('\n')+'\n\n### Strengths\n'+L.res.strengths.map(x=>'- '+x).join('\n')+'\n\n### Weaknesses\n'+L.res.weaknesses.map(x=>'- '+x).join('\n')+'\n\n### Recommendations\n'+L.res.recommendations.map(x=>'- '+x).join('\n')+'\n\n### Optimized prompt\n\n```text\n'+L.optimized+'\n```\n';
 download('prompt_report.md',m,'text/markdown')};

/* ---------- reel: generated "video" with custom media controls ---------- */
const DUR=32;
function drawReel(ctx,t,k){const W=960,H=540;ctx.setTransform(k,0,0,k,0,0);t=Math.min(t,DUR-.001);
 const g=ctx.createLinearGradient(0,0,W,H);g.addColorStop(0,'#090b0a');g.addColorStop(1,'#0f2a1b');ctx.fillStyle=g;ctx.fillRect(0,0,W,H);
 ctx.lineWidth=1.2;for(let i=0;i<9;i++){ctx.strokeStyle=`rgba(63,207,142,${.08+.04*Math.sin(t+i)})`;ctx.beginPath();for(let x=0;x<=W;x+=24){const y=70+i*56+Math.sin(x*.012+t*.8+i)*22;x?ctx.lineTo(x,y):ctx.moveTo(x,y)}ctx.stroke()}
 const si=Math.min(3,Math.floor(t/8)),lt=(t%8)/8,N=['01  WRITE','02  EVALUATE','03  IMPROVE','04  MASTER'];
 ctx.font='600 15px "JetBrains Mono",monospace';ctx.textAlign='left';N.forEach((n,i)=>{ctx.fillStyle=i===si?'#ff3b5c':'rgba(228,238,232,.3)';ctx.fillText(n,44+i*170,52)});
 ctx.fillStyle='rgba(255,255,255,.1)';ctx.fillRect(44,66,W-88,2);ctx.fillStyle='#dc143c';ctx.fillRect(44,66,(W-88)*t/DUR,2);
 const e=ease(clamp(lt*1.6)),KS=KEYS.length;
 if(si===0){const L=['You are an HR manager.','Create five interview questions','for a Python developer.','Return a numbered list.','Use a professional tone.'],all=L.join('\n'),n=Math.floor(all.length*clamp(lt*1.25));
  const sh=all.slice(0,n).split('\n');ctx.font='500 34px "JetBrains Mono",monospace';sh.forEach((l,i)=>{ctx.fillStyle=i%2?'#e4eee8':'#3fcf8e';ctx.fillText(l+(i===sh.length-1&&Math.floor(t*3)%2?'▍':''),90,190+i*62)})}
 if(si===1){const S=[8,7,6,4,7,5],cx=480,cy=315,R=170,pt=(i,v)=>{const a=-Math.PI/2+i*2*Math.PI/KS;return[cx+Math.cos(a)*R*v/10,cy+Math.sin(a)*R*v/10]};
  ctx.strokeStyle='rgba(120,190,150,.25)';[.33,.66,1].forEach(f=>{ctx.beginPath();KEYS.forEach((_,i)=>{const[x,y]=pt(i,10*f);i?ctx.lineTo(x,y):ctx.moveTo(x,y)});ctx.closePath();ctx.stroke()});
  ctx.fillStyle='rgba(63,207,142,.28)';ctx.strokeStyle='#3fcf8e';ctx.lineWidth=3;ctx.beginPath();S.forEach((v,i)=>{const[x,y]=pt(i,v*e);i?ctx.lineTo(x,y):ctx.moveTo(x,y)});ctx.closePath();ctx.fill();ctx.stroke();
  ctx.font='600 17px "Space Grotesk",sans-serif';ctx.fillStyle='#e4eee8';ctx.textAlign='center';KEYS.forEach((k,i)=>{const[x,y]=pt(i,12);ctx.fillText(k,x,y+5)});
  ctx.textAlign='left';ctx.font='800 64px Syne,sans-serif';ctx.fillStyle='#dc143c';ctx.fillText(Math.round(62*e),90,300);ctx.font='500 16px "JetBrains Mono",monospace';ctx.fillStyle='#7e9388';ctx.fillText('/100  baseline',90,330)}
 if(si===2){const row=(y,lb,v,c)=>{ctx.fillStyle='rgba(255,255,255,.08)';ctx.fillRect(90,y,780,46);ctx.fillStyle=c;ctx.fillRect(90,y,780*v/100,46);ctx.fillStyle='#fff';ctx.font='700 22px "Space Grotesk",sans-serif';ctx.textAlign='left';ctx.fillText(lb+'  '+Math.round(v),104,y+31)};
  row(200,'Your draft',62,'#8f0f26');row(290,'Optimized',62+30*e,'#1f7a45');ctx.font='500 18px "JetBrains Mono",monospace';ctx.fillStyle='#7e9388';ctx.fillText('+ role   + constraints   + format   + tone',90,400)}
 if(si===3){const cx=480,cy=300,R=120,f=ease(clamp(lt*1.5));ctx.lineWidth=16;ctx.lineCap='round';ctx.strokeStyle='rgba(255,255,255,.08)';ctx.beginPath();ctx.arc(cx,cy,R,0,6.283);ctx.stroke();
  ctx.strokeStyle='#3fcf8e';ctx.shadowColor='#3fcf8e';ctx.shadowBlur=24;ctx.beginPath();ctx.arc(cx,cy,R,-Math.PI/2,-Math.PI/2+6.283*.92*f);ctx.stroke();ctx.shadowBlur=0;
  ctx.textAlign='center';ctx.fillStyle='#e4eee8';ctx.font='800 84px Syne,sans-serif';ctx.fillText(Math.round(92*f),cx,cy+26);ctx.font='700 20px "JetBrains Mono",monospace';ctx.fillStyle='#ff3b5c';ctx.fillText('PROMPT MASTERED',cx,cy+R+62);
  for(let i=0;i<3;i++){const p=((t*.6+i/3)%1);ctx.strokeStyle=`rgba(220,20,60,${.5*(1-p)})`;ctx.lineWidth=2;ctx.beginPath();ctx.arc(cx,cy,R+20+p*120,0,6.283);ctx.stroke()}}
 return si}
/* preview tile (muted, looping) */
const pv=$('#reelPrev'),pctx=pv.getContext('2d');let pf=0;updaters.push((dt,now)=>{if(++pf%2)return;if($('#reelModal').classList.contains('on'))return;drawReel(pctx,(now/1000)%DUR,pv.width/960)});
/* follow label with spring */
const tile=$('#reelTile'),fol=$('#follow'),fX=new Spring(0,{mass:.1,k:170,c:26,m:.1}),fY=new Spring(0,{k:170,c:26,m:.1}),fO=new Spring(0,{k:170,c:26,m:.1});fX.o=P.orb;fY.o=P.orb;fO.o=P.orb;
tile.addEventListener('pointermove',e=>{const b=tile.getBoundingClientRect();fO.t=1;fX.t=e.clientX-b.left;fY.t=e.clientY-b.top});tile.addEventListener('pointerleave',()=>fO.t=0);
updaters.push(dt=>{if(fX.settled&&fY.settled&&fO.settled)return;fX.step(dt);fY.step(dt);fO.step(dt);fol.style.opacity=clamp(fO.x);fol.style.transform=`translate(${fX.x}px,${fY.x}px)`});
/* modal player */
const rm={t:0,playing:true,muted:false,ac:null,last:-1,open:false};const cv=$('#reelCv'),cx2=cv.getContext('2d'),box=$('#reelBox'),mod=$('#reelModal'),back=$('#rmBack');
const ICON={play:'<svg viewBox="0 0 24 24"><path d="M7 4l13 8-13 8z"/></svg>',pause:'<svg viewBox="0 0 24 24"><path d="M6 4h4v16H6zM14 4h4v16h-4z"/></svg>',vol:'<svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 4V5L7 9zM15 8a5 5 0 010 8" stroke="#fff" fill="none" stroke-width="2"/></svg>',mute:'<svg viewBox="0 0 24 24"><path d="M3 9v6h4l5 4V5L7 9z"/><path d="M16 9l5 6M21 9l-5 6" stroke="#fff" stroke-width="2"/></svg>'};
const A0='inset(43.5% 43.5% 33.5% 43.5%)';
function uiRm(){$('#rmPlay').innerHTML=rm.playing?ICON.pause:ICON.play;$('#rmMute').innerHTML=rm.muted?ICON.mute:ICON.vol;const s=Math.floor(rm.t),f=n=>Math.floor(n/60)+':'+String(n%60).padStart(2,'0');$('#rmTime').textContent=f(s)+' / '+f(DUR);$('#rmRange').value=rm.t*100}
function blip(i){if(rm.muted)return;try{rm.ac=rm.ac||new (window.AudioContext||window.webkitAudioContext)();const o=rm.ac.createOscillator(),g=rm.ac.createGain();o.type='sine';o.frequency.value=[196,262,330,523][i];g.gain.setValueAtTime(.0001,rm.ac.currentTime);g.gain.exponentialRampToValueAtTime(.12,rm.ac.currentTime+.04);g.gain.exponentialRampToValueAtTime(.0001,rm.ac.currentTime+.7);o.connect(g).connect(rm.ac.destination);o.start();o.stop(rm.ac.currentTime+.8)}catch(e){}}
updaters.push(dt=>{if(!rm.open)return;if(rm.playing){rm.t+=dt;if(rm.t>=DUR)rm.t=0}const si=drawReel(cx2,rm.t,1);if(si!==rm.last){if(rm.last!==-1&&rm.playing)blip(si);rm.last=si}uiRm()});
tile.addEventListener('click',()=>{rm.open=true;rm.t=0;rm.playing=true;rm.last=-1;mod.classList.add('on');back.style.opacity=1;
 box.animate([{clipPath:A0,opacity:0},{clipPath:'inset(0 0 0 0)',opacity:1}],{duration:RM?1:900,easing:'cubic-bezier(.16,1,.3,1)',fill:'forwards'});uiRm()});
function closeReel(){if(!rm.open)return;rm.open=false;back.style.opacity=0;const a=box.animate([{clipPath:'inset(0 0 0 0)',opacity:1,offset:0},{opacity:1,offset:.7},{clipPath:A0,opacity:0,offset:1}],{duration:RM?1:800,easing:'cubic-bezier(.5,0,.2,1)',fill:'forwards'});a.onfinish=()=>mod.classList.remove('on')}
$('#rmX').onclick=closeReel;back.onclick=closeReel;document.addEventListener('keydown',e=>{if(e.key==='Escape')closeReel()});
$('#rmPlay').onclick=()=>{rm.playing=!rm.playing;uiRm()};$('#rmMute').onclick=()=>{rm.muted=!rm.muted;uiRm()};
$('#rmRange').addEventListener('input',e=>{rm.t=e.target.value/100;rm.last=Math.floor(rm.t/8);uiRm()});

/* ---------- nav / dial panel ---------- */
$$('[data-go]').forEach(b=>b.onclick=()=>sc.scrollTo({top:$('#'+b.dataset.go).offsetTop-(b.dataset.go==='hero'?0:40)}));
$('#dialBtn').onclick=()=>$('#dial').classList.toggle('on');
(function dialUI(){const rows=[['Caret stiffness',P.caret,'k',50,1200,10],['Caret damping',P.caret,'c',5,80,1],['Caret mass',P.caret,'m',.1,2,.05],['Orb stiffness',P.orb,'k',20,400,1],['Orb damping',P.orb,'c',2,40,.5],['Orb mass',P.orb,'m',.05,1,.05],['Editor font',P,'fs',12,28,1],['Glow',P,'glow',0,2,.1]],box=$('#dialRows');
 rows.forEach(([n,o,k,mn,mx,st])=>{const r=el('div','dr');r.innerHTML='<label>'+n+'</label><input type="range" min="'+mn+'" max="'+mx+'" step="'+st+'" value="'+o[k]+'"><output>'+o[k]+'</output>';
  const i=$('input',r),out=$('output',r);i.oninput=()=>{o[k]=+i.value;out.textContent=o[k];if(k==='fs')root.style.setProperty('--fs',o.fs+'px');if(k==='glow')root.style.setProperty('--glow',o.glow);carets.forEach(c=>c.update())};box.appendChild(r)});
 const t=el('div','dr');t.innerHTML='<label>Label input</label><select><option>text</option><option>password</option></select>';$('select',t).onchange=e=>{lab.type=e.target.value;carets.forEach(c=>c.update())};box.appendChild(t);
 const m=el('div','dr');m.innerHTML='<label>Reduce motion</label><input type="checkbox" '+(RM?'checked':'')+'>';$('input',m).onchange=e=>RM=e.target.checked;box.appendChild(m)})();
try{if(IN_COLAB)google.colab.output.setIframeHeight(0,true,{minHeight:940})}catch(e){}
})();
</script>

"""
UI = (UI.replace("__SCENARIOS_JSON__", json.dumps(SCENARIOS))
        .replace("__CRITERIA_JSON__", json.dumps(CRITERIA)))

# ---------------- web link (Flask) ----------------
import threading
from flask import Flask, request, jsonify, Response
from werkzeug.serving import make_server
from google.colab.output import eval_js

PORT = 8765
PAGE = ('<!doctype html><html><head><meta charset="utf-8">'
        '<meta name="viewport" content="width=device-width,initial-scale=1">'
        '<title>PROMPT·LAB</title></head><body style="margin:0;background:#050706">' + UI + '</body></html>')

app = Flask(__name__)

@app.get("/")
def _index():
    return Response(PAGE, mimetype="text/html")

@app.post("/api/<name>")
def _api(name):
    fn = API.get(name)
    if fn is None:
        return jsonify({"error": "unknown endpoint"}), 404
    return jsonify(fn(*(request.get_json(force=True).get("args") or [])))

if globals().get("_srv"):          # re-running the cell: stop the old server
    try: _srv.shutdown()
    except Exception: pass
    time.sleep(0.5)
_srv = make_server("0.0.0.0", PORT, app, threaded=True)
threading.Thread(target=_srv.serve_forever, daemon=True).start()

URL = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print("🌐 PROMPT·LAB link:", URL)
display(HTML(f'<a href="{URL}" target="_blank" style="display:inline-block;padding:12px 22px;'
             f'border-radius:18px 6px 18px 6px;background:linear-gradient(120deg,#1f7a45,#8f0f26);'
             f'color:#fff;font:600 14px system-ui;text-decoration:none;letter-spacing:.06em">'
             f'⚡ OPEN PROMPT·LAB IN A NEW TAB</a>'))
display(HTML(UI))   # inline copy right here in the notebook
